# Selección temporal de variables para el modelo PD

Cuaderno de experimento reproducible. Evalúa qué entradas son necesarias para predecir `default_24m` sin utilizar los años 2021–2022 durante la selección.

## tl;dr

La regresión logística con cinco variables de originación —FICO, DTI, CLTV, tipo de interés y número de prestatarios— supera en AUC al conjunto completo de 18 variables en los cuatro cortes 2017–2020. En 2020 mejora el AUC de 0,7121 a 0,7192; el Brier y el log-loss empeoran solo un 0,07 % y un 0,23 %, respectivamente. Se propone este núcleo para el producto mínimo.

## Contexto y método

**Decisión.** Elegir el menor contrato de entrada que conserve capacidad discriminante y calibración suficientes para una demostración académica productivizada.

**Modelo fijo.** Regresión logística del proyecto, con imputación, escalado, ponderación de clases y calibración sigmoide. Solo cambian las columnas de entrada. La selección es específica de esta logística; HGB se comparará después usando el mismo contrato congelado.

**Diseño temporal.** Cuatro validaciones encadenadas: desarrollo histórico, un año de calibración y el año siguiente de validación. Los años 2021–2022 no intervienen en perfiles, correlaciones ni modelos de selección.

### Supuestos clave

- La unidad es un préstamo anonimizado preparado por el pipeline Freddie. Cada año representa solo Q1 y un muestreo determinista de hasta 50.000 préstamos elegibles; no es la cohorte anual completa.
- El objetivo es impago a 24 meses según la definición registrada en `configs/model.json`.
- La selección prioriza parsimonia si las pérdidas frente al modelo completo quedan dentro de tolerancias fijadas antes del test final.
- El compacto no contiene identificador original; por ello la unicidad del préstamo depende del proceso de preparación y no puede revalidarse contra el ZIP desde este cuaderno.

In [1]:
from pathlib import Path
import hashlib
import platform
import warnings
import numpy as np
import pandas as pd
import sklearn
from pandas.errors import DtypeWarning

from src.data_access import read_csv_zst
from src.metrics import classification_metrics
from src.pd_model import PDConfig, _calibrate, _logistic, _threshold

ROOT = Path.cwd()
DATA_PATH = ROOT / 'freddie-analysis.csv.zst'
EXPECTED_SHA256 = '468a7a4e0b80b3dec1722b27a549f0c3a937ddc1cf2c99aa7f77ce51f4ae0e0e'
SEED = 20260819
TEST_YEARS = {2021, 2022}
print({'python': platform.python_version(), 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__, 'seed': SEED})

{'python': '3.12.13', 'pandas': '2.3.3', 'scikit_learn': '1.8.0', 'seed': 20260819}


## Datos

### 1. Verificar identidad, cobertura y campos esenciales

In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert DATA_PATH.is_file(), f'No existe {DATA_PATH}'
assert sha256(DATA_PATH) == EXPECTED_SHA256
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DtypeWarning)  # columna de fecha mixta; no es predictora
    all_rows = pd.concat(read_csv_zst(DATA_PATH, chunksize=100_000), ignore_index=True)
assert len(all_rows) == 399_800
frame = all_rows.loc[~all_rows['cohort_year'].isin(TEST_YEARS)].copy()
del all_rows
assert len(frame) == 299_845
assert not TEST_YEARS.intersection(frame['cohort_year'].unique())
assert set(frame['default_24m'].dropna().unique()) == {0, 1}

core_features = [
    'origination_fico',
    'original_dti',
    'original_cltv',
    'original_interest_rate',
    'number_of_borrowers',
]
quality = pd.DataFrame({
    'missing_rate': frame[core_features].isna().mean(),
    'minimum': frame[core_features].min(),
    'median': frame[core_features].median(),
    'maximum': frame[core_features].max(),
}).round(4)
cohorts = frame.groupby('cohort_year', observed=True)['default_24m'].agg(rows='size', events='sum')
cohorts['event_rate'] = (cohorts['events'] / cohorts['rows']).round(6)
print(f'Fichero: {DATA_PATH.name}; filas totales verificadas: 399,800; ' + f'filas usadas en selección: {len(frame):,}; SHA-256: {sha256(DATA_PATH)}')
print('\nCobertura temporal:')
print(cohorts.to_string())
print('\nPerfil de las cinco variables:')
print(quality.to_string())
print('\nComprobaciones de extremos y ausencias:')
print({
    'fico_missing': int(frame['origination_fico'].isna().sum()),
    'dti_missing': int(frame['original_dti'].isna().sum()),
    'cltv_gt_200': int(frame['original_cltv'].gt(200).sum()),
    'cltv_gt_200_events': int(frame.loc[frame['original_cltv'].gt(200), 'default_24m'].sum()),
})

Fichero: freddie-analysis.csv.zst; filas totales verificadas: 399,800; filas usadas en selección: 299,845; SHA-256: 468a7a4e0b80b3dec1722b27a549f0c3a937ddc1cf2c99aa7f77ce51f4ae0e0e

Cobertura temporal:
              rows  events  event_rate
cohort_year                           
2015         49985     164    0.003281
2016         49981     307    0.006142
2017         49979     420    0.008404
2018         49970     310    0.006204
2019         49955    2201    0.044060
2020         49975    1744    0.034897

Perfil de las cinco variables:
                        missing_rate  minimum   median  maximum
origination_fico              0.0002  428.000  757.000  839.000
original_dti                  0.0429    1.000   36.000   51.000
original_cltv                 0.0000    3.000   79.000  610.000
original_interest_rate        0.0000    2.125    4.125    6.875
number_of_borrowers           0.0000    1.000    1.000    5.000

Comprobaciones de extremos y ausencias:
{'fico_missing': 58, 'dti_mis

### 2. Definir candidatos y cortes sin test

In [3]:
full_numeric = (
    'original_interest_rate', 'original_upb', 'original_loan_term', 'original_ltv',
    'original_cltv', 'number_of_borrowers', 'original_dti', 'origination_fico',
    'mortgage_insurance_percentage',
)
full_categorical = (
    'first_time_home_buyer', 'loan_purpose', 'property_type', 'number_of_units',
    'occupancy_status', 'property_state', 'amortization_type',
    'mortgage_insurance_type', 'high_balance_loan',
)
core_5 = tuple(core_features)
candidates = {
    'full_18': (full_numeric, full_categorical),
    'clean_12': (
        tuple(name for name in full_numeric if name not in {'original_ltv', 'mortgage_insurance_percentage'}),
        tuple(name for name in full_categorical if name not in {
            'number_of_units', 'amortization_type', 'mortgage_insurance_type', 'high_balance_loan'
        }),
    ),
    'core_5': (core_5, ()),
    'core_state_6': (core_5, ('property_state',)),
    'product_7': (core_5, ('occupancy_status', 'loan_purpose')),
    'product_state_8': (core_5, ('occupancy_status', 'loan_purpose', 'property_state')),
}
folds = {
    2017: ((2015,), 2016, 2017),
    2018: ((2015, 2016), 2017, 2018),
    2019: ((2015, 2016, 2017), 2018, 2019),
    2020: ((2015, 2016, 2017, 2018), 2019, 2020),
}
assert all(not TEST_YEARS.intersection((*development, calibration, validation)) for development, calibration, validation in folds.values())
print(pd.DataFrame([
    {'validation': name, 'development': ', '.join(map(str, development)),      'calibration': calibration}
    for name, (development, calibration, _) in folds.items()
]).to_string(index=False))

 validation            development  calibration
       2017                   2015         2016
       2018             2015, 2016         2017
       2019       2015, 2016, 2017         2018
       2020 2015, 2016, 2017, 2018         2019


### 3. Perfilar las variables descartables

In [4]:
discardable = [
    'original_upb', 'original_loan_term', 'original_ltv',
    'mortgage_insurance_percentage', 'first_time_home_buyer', 'loan_purpose',
    'property_type', 'number_of_units', 'occupancy_status', 'property_state',
    'amortization_type', 'mortgage_insurance_type', 'high_balance_loan',
]
profile_rows = []
for name in discardable:
    values = frame[name]
    profile_rows.append({
        'feature': name,
        'missing_rate': values.isna().mean(),
        'distinct': values.nunique(dropna=True),
        'top_value_share': values.value_counts(dropna=False, normalize=True).iloc[0],
    })
discard_profile = pd.DataFrame(profile_rows).set_index('feature')
print(discard_profile.round(4).to_string())
print({
    'spearman_ltv_cltv': round(frame[['original_ltv', 'original_cltv']].corr(method='spearman').iloc[0, 1], 4),
    'pearson_mi_ltv': round(frame[['mortgage_insurance_percentage', 'original_ltv']].corr().iloc[0, 1], 4),
    'pearson_mi_cltv': round(frame[['mortgage_insurance_percentage', 'original_cltv']].corr().iloc[0, 1], 4),
})

                               missing_rate  distinct  top_value_share
feature                                                               
original_upb                            0.0       840           0.0118
original_loan_term                      0.0       176           0.7663
original_ltv                            0.0       197           0.2062
mortgage_insurance_percentage           0.0        18           0.7411
first_time_home_buyer                   0.0         3           0.8178
loan_purpose                            0.0         3           0.4674
property_type                           0.0         5           0.6396
number_of_units                         0.0         5           0.9758
occupancy_status                        0.0         3           0.8758
property_state                          0.0        54           0.1408
amortization_type                       0.0         1           1.0000
mortgage_insurance_type                 0.0         2           0.7411
high_b

## Resultados

### 4. Comparar los seis contratos de entrada

In [5]:
def evaluate_candidate(numeric, categorical, development_years, calibration_year, validation_year):
    config = PDConfig(tuple(numeric), tuple(categorical), SEED)
    development = frame.loc[frame['cohort_year'].isin(development_years)]
    calibration = frame.loc[frame['cohort_year'].eq(calibration_year)]
    validation = frame.loc[frame['cohort_year'].eq(validation_year)]
    model = _logistic(config).fit(development[list(config.features)], development[config.target])
    calibrated = _calibrate(model, calibration[list(config.features)], calibration[config.target])
    probability = calibrated.predict_proba(validation[list(config.features)])[:, 1]
    threshold = _threshold(validation[config.target], probability)
    return classification_metrics(validation[config.target], probability, threshold)

rows = []
for validation_year, fold in folds.items():
    for candidate, (numeric, categorical) in candidates.items():
        metrics = evaluate_candidate(numeric, categorical, *fold)
        rows.append({'validation_year': validation_year, 'candidate': candidate, **metrics})
results = pd.DataFrame(rows)
shown = results[[
    'validation_year', 'candidate', 'roc_auc', 'pr_auc', 'ks', 'brier', 'log_loss',
    'calibration_intercept', 'calibration_slope'
]].copy()
print(shown.round(5).to_string(index=False))

 validation_year       candidate  roc_auc  pr_auc      ks   brier  log_loss  calibration_intercept  calibration_slope
            2017         full_18  0.75143 0.02482 0.39324 0.00828   0.04529               -0.16972            0.93082
            2017        clean_12  0.74927 0.02577 0.38908 0.00827   0.04531               -0.16997            0.93094
            2017          core_5  0.78808 0.03327 0.43811 0.00825   0.04390                0.54669            1.09609
            2017    core_state_6  0.75506 0.02851 0.38440 0.00827   0.04511               -0.16778            0.92837
            2017       product_7  0.78814 0.03180 0.44340 0.00826   0.04401                0.62572            1.11204
            2017 product_state_8  0.75973 0.02663 0.39416 0.00827   0.04503                0.26368            1.01647
            2018         full_18  0.75507 0.02354 0.40312 0.00625   0.03619               -1.63792            0.70555
            2018        clean_12  0.75448 0.02268 0.3989

### 5. Aplicar criterios fijados antes del test final

In [6]:
comparison = results.pivot(index='validation_year', columns='candidate')
baseline = results.loc[results['candidate'].eq('full_18')].set_index('validation_year')
core = results.loc[results['candidate'].eq('core_5')].set_index('validation_year')
deltas = pd.DataFrame({
    'auc_delta': core['roc_auc'] - baseline['roc_auc'],
    'pr_auc_delta': core['pr_auc'] - baseline['pr_auc'],
    'ks_delta': core['ks'] - baseline['ks'],
    'brier_relative': core['brier'] / baseline['brier'] - 1,
    'log_loss_relative': core['log_loss'] / baseline['log_loss'] - 1,
})
performance_ok = (
    (deltas['auc_delta'] >= -0.01)
    & (deltas['pr_auc_delta'] >= -0.005)
    & (deltas['ks_delta'] >= -0.02)
    & (deltas['brier_relative'] <= 0.02)
    & (deltas['log_loss_relative'] <= 0.02)
).all()
calibration_ok = (
    -0.25 <= core.loc[2020, 'calibration_intercept'] <= 0.25
    and 0.8 <= core.loc[2020, 'calibration_slope'] <= 1.2
)
assert performance_ok and calibration_ok
assert (deltas['auc_delta'] > 0).all()
print(deltas.round(6).to_string())
print(f'\nAceptación core_5: rendimiento={performance_ok}; calibración_2020={calibration_ok}')

                 auc_delta  pr_auc_delta  ks_delta  brier_relative  log_loss_relative
validation_year                                                                      
2017              0.036642      0.008451  0.044875       -0.003242          -0.030652
2018              0.044866      0.004890  0.060716       -0.020319          -0.049565
2019              0.013053      0.001923  0.012053       -0.003154          -0.013438
2020              0.007137      0.001621  0.013703        0.000672           0.002310

Aceptación core_5: rendimiento=True; calibración_2020=True


### 6. Ablación: retirar una variable del núcleo

In [7]:
ablation_rows = []
for validation_year, fold in folds.items():
    core_metrics = core.loc[validation_year]
    for removed in core_5:
        reduced = tuple(name for name in core_5 if name != removed)
        metrics = evaluate_candidate(reduced, (), *fold)
        ablation_rows.append({
            'validation_year': validation_year,
            'removed': removed,
            'auc_delta': metrics['roc_auc'] - core_metrics['roc_auc'],
            'log_loss_relative': metrics['log_loss'] / core_metrics['log_loss'] - 1,
        })
ablation = pd.DataFrame(ablation_rows)
auc_ablation = ablation.pivot(index='removed', columns='validation_year', values='auc_delta')
interest_2019 = ablation.loc[
    ablation['removed'].eq('original_interest_rate')
    & ablation['validation_year'].eq(2019), 'log_loss_relative'
].iloc[0]
state = results.loc[results['candidate'].eq('core_state_6')].set_index('validation_year')
product = results.loc[results['candidate'].eq('product_7')].set_index('validation_year')
assert (state['roc_auc'] < core['roc_auc']).all()
assert interest_2019 > 0.04
print('Cambio de AUC al retirar cada variable (negativo = empeora):')
print(auc_ablation.round(5).to_string())
print(f'\nSin interés, deterioro relativo del log-loss en 2019: {interest_2019:.2%}')
print('Incremento de AUC de ocupación + finalidad frente a core_5:')
print((product['roc_auc'] - core['roc_auc']).round(5).to_string())

Cambio de AUC al retirar cada variable (negativo = empeora):
validation_year            2017     2018     2019     2020
removed                                                   
number_of_borrowers    -0.01469 -0.02774 -0.00637 -0.00254
original_cltv          -0.00420 -0.00006 -0.00686 -0.00625
original_dti           -0.00378 -0.00929 -0.01569 -0.02134
original_interest_rate  0.00029 -0.00566 -0.00312 -0.00455
origination_fico       -0.08446 -0.07269 -0.06376 -0.03573

Sin interés, deterioro relativo del log-loss en 2019: 4.16%
Incremento de AUC de ocupación + finalidad frente a core_5:
validation_year
2017    0.00007
2018    0.00435
2019   -0.00241
2020    0.00106


### 7. Reincorporar individualmente cada variable descartada

In [8]:
discarded_numeric = (
    'original_upb', 'original_loan_term', 'original_ltv', 'mortgage_insurance_percentage',
)
discarded_categorical = (
    'first_time_home_buyer', 'loan_purpose', 'property_type', 'number_of_units',
    'occupancy_status', 'property_state', 'amortization_type',
    'mortgage_insurance_type', 'high_balance_loan',
)
addback_rows = []
for validation_year, fold in folds.items():
    core_metrics = core.loc[validation_year]
    for feature in discarded_numeric:
        metrics = evaluate_candidate(core_5 + (feature,), (), *fold)
        addback_rows.append({
            'validation_year': validation_year,
            'feature': feature,
            'auc_delta': metrics['roc_auc'] - core_metrics['roc_auc'],
            'log_loss_relative': metrics['log_loss'] / core_metrics['log_loss'] - 1,
        })
    for feature in discarded_categorical:
        metrics = evaluate_candidate(core_5, (feature,), *fold)
        addback_rows.append({
            'validation_year': validation_year,
            'feature': feature,
            'auc_delta': metrics['roc_auc'] - core_metrics['roc_auc'],
            'log_loss_relative': metrics['log_loss'] / core_metrics['log_loss'] - 1,
        })
addback = pd.DataFrame(addback_rows)
print('Cambio de AUC al añadir cada variable al núcleo:')
print(addback.pivot(index='feature', columns='validation_year', values='auc_delta').round(5).to_string())
print('\nCambio relativo de log-loss al añadirla (negativo = mejora):')
print(addback.pivot(index='feature', columns='validation_year', values='log_loss_relative').round(5).to_string())


Cambio de AUC al añadir cada variable al núcleo:
validation_year                   2017     2018     2019     2020
feature                                                          
amortization_type              0.00002  0.00000  0.00001 -0.00000
first_time_home_buyer          0.00006 -0.00044  0.00000 -0.00054
high_balance_loan              0.00065 -0.00628 -0.00374 -0.00299
loan_purpose                   0.00196  0.00229  0.00061  0.00086
mortgage_insurance_percentage  0.00039  0.00149  0.00018 -0.00046
mortgage_insurance_type        0.00025  0.00117  0.00004 -0.00059
number_of_units                0.00039 -0.00047 -0.00565 -0.00362
occupancy_status              -0.00143  0.00233 -0.00288  0.00030
original_loan_term            -0.00294 -0.00009 -0.00019  0.00028
original_ltv                   0.00006  0.00004  0.00001 -0.00000
original_upb                  -0.00358  0.00075 -0.00421 -0.00680
property_state                -0.03302 -0.05279 -0.01007 -0.00586
property_type              

## Conclusiones

1. **Conservar** FICO: es la señal individual más estable; retirarla reduce el AUC entre aproximadamente 0,036 y 0,084.
2. **Conservar** DTI: aporta capacidad de pago y su retirada empeora especialmente 2019–2020.
3. **Conservar** CLTV: resume apalancamiento y garantía; evita mantener LTV, altamente redundante.
4. **Conservar** tipo de interés: aunque su efecto sobre AUC es moderado, retirarlo degrada materialmente el log-loss de 2019.
5. **Conservar** número de prestatarios: mejora de forma clara los cortes tempranos y añade una dimensión simple de estructura del préstamo.
6. **Descartar** ocupación y finalidad: añaden solo 0,0011 de AUC en 2020 y no mejoran todos los años.
7. **Descartar** estado: sus 54 categorías reducen el AUC en los cuatro cortes y dificultan la defensa y la entrada manual.
8. **Add-back individual:** ninguna de las trece variables descartadas aporta una mejora grande y estable. Finalidad suma como máximo 0,0023 de AUC, con log-loss mixto; las demás son casi neutras o empeoran varios cortes. Se excluyen por parsimonia, no porque se afirme que su señal poblacional sea exactamente cero.
9. **Control de dominio pendiente:** existen 28 CLTV superiores a 200 y ninguno es evento; antes de cerrar la API se comprobará si excluirlos o limitar el rango cambia las métricas.

**Decisión provisional para la logística del MVP:** el contrato tendrá cinco entradas. El holdout 2021–2022 se evaluará una sola vez después de congelar el código y las bandas de riesgo. Como esos años aparecieron en informes anteriores, se declararán como holdout confirmatorio previamente expuesto, no como test completamente virgen.